# Phase 18 — Augmented LightGBM Training

Retrains the Phase 11 lead scorer on a **combined corpus** of real Retailrocket
sessions + CTGAN-generated synthetic sessions, then compares v1 vs v2 across
AUC, Precision@K, and Recall@K metrics.

**Inputs:**
- `data/augmented_training.parquet` — produced by `make build-augmented-dataset`
- `data/augmented_test.parquet` — held-out stratified test set
- `models/lead_scorer_lgbm.pkl` — Phase 11 v1 baseline

**Output:** `models/lead_scorer_lgbm_v2.pkl` + `docs/model_card_v2.md`

**Prerequisite:** `make build-augmented-dataset` must complete before running this notebook.

In [ ]:
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings('ignore', category=UserWarning)

sys.path.insert(0, str(Path('..').resolve()))
from src.scoring.ml_scorer import _FEATURE_COLS

RANDOM_STATE   = 42
TRAIN_PATH     = Path('../data/augmented_training.parquet')
TEST_PATH      = Path('../data/augmented_test.parquet')
V1_MODEL_PATH  = Path('../models/lead_scorer_lgbm.pkl')
V2_MODEL_PATH  = Path('../models/lead_scorer_lgbm_v2.pkl')
MODEL_CARD_PATH = Path('../docs/model_card_v2.md')

print('LightGBM version:', lgb.__version__)
print('Feature cols    :', _FEATURE_COLS)

## Cell 1 — Load augmented dataset

In [ ]:
df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

print(f'Training set : {len(df_train):>10,} rows')
print(f'Test set     : {len(df_test):>10,} rows')

# Class distribution breakdown by is_synthetic flag
if 'is_synthetic' in df_train.columns:
    print('\n--- Training set provenance ---')
    prov = df_train.groupby('is_synthetic')['converted'].agg(['count', 'sum', 'mean'])
    prov.index = prov.index.map({0: 'real', 1: 'synthetic'})
    prov.columns = ['total', 'converted', 'conversion_rate']
    prov['conversion_rate'] = prov['conversion_rate'].map('{:.2%}'.format)
    display(prov)

print(f'\nOverall conversion rate (train): {df_train["converted"].mean():.2%}')
print(f'Overall conversion rate (test) : {df_test["converted"].mean():.2%}')

## Cell 2 — Baseline v1 recall on augmented test set

In [ ]:
def recall_at_k(y_true, y_prob, k_pct=0.10):
    """Fraction of true positives captured in the top-K% predicted sessions."""
    k = max(1, int(len(y_true) * k_pct))
    top_k_idx = np.argsort(y_prob)[::-1][:k]
    captured = np.array(y_true)[top_k_idx].sum()
    total_positives = np.array(y_true).sum()
    return captured / total_positives if total_positives > 0 else 0.0

def precision_at_k(y_true, y_prob, k_pct=0.10):
    k = max(1, int(len(y_true) * k_pct))
    top_k_idx = np.argsort(y_prob)[::-1][:k]
    return np.array(y_true)[top_k_idx].mean()


X_test = df_test[_FEATURE_COLS]
y_test = df_test['converted']

v1_model = joblib.load(V1_MODEL_PATH)
v1_prob  = v1_model.predict_proba(X_test)[:, 1]

v1_auc        = roc_auc_score(y_test, v1_prob)
v1_recall_k   = recall_at_k(y_test, v1_prob)
v1_precision_k = precision_at_k(y_test, v1_prob)

print('=== Phase 11 v1 Baseline (on augmented test set) ===')
print(f'  ROC-AUC          : {v1_auc:.4f}')
print(f'  Recall@10%       : {v1_recall_k:.4f}')
print(f'  Precision@10%    : {v1_precision_k:.4f}')

## Cell 3 — Train augmented LightGBM v2

In [ ]:
X_train = df_train[_FEATURE_COLS]
y_train = df_train['converted']

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'Training: {len(X_train):,} rows | positive rate: {pos/len(y_train):.2%}')
print(f'scale_pos_weight: {scale_pos_weight:.1f}')

v2_model = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    verbose=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    v2_model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1
)
print(f'\nCV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per fold  : {np.round(cv_scores, 4)}')

v2_model.fit(X_train, y_train)
print('\nFull training complete.')

## Cell 4 — v1 vs v2 comparison

In [ ]:
v2_prob = v2_model.predict_proba(X_test)[:, 1]

v2_auc         = roc_auc_score(y_test, v2_prob)
v2_recall_k    = recall_at_k(y_test, v2_prob)
v2_precision_k = precision_at_k(y_test, v2_prob)

comparison = pd.DataFrame({
    'Metric': ['ROC-AUC', 'Recall@10%', 'Precision@10%'],
    'v1 (baseline)': [v1_auc, v1_recall_k, v1_precision_k],
    'v2 (augmented)': [v2_auc, v2_recall_k, v2_precision_k],
})
comparison['delta'] = comparison['v2 (augmented)'] - comparison['v1 (baseline)']
comparison['delta_pct_pp'] = comparison['delta'] * 100

display(comparison.set_index('Metric').round(4))

recall_improvement_pp = (v2_recall_k - v1_recall_k) * 100
TARGET_IMPROVEMENT_PP = 5.0

if recall_improvement_pp >= TARGET_IMPROVEMENT_PP:
    print(f'\n✓ Recall@10% improved by {recall_improvement_pp:.1f}pp — target met ({TARGET_IMPROVEMENT_PP:.0f}pp).')
else:
    print(
        f'\n⚠ Recall@10% improved by only {recall_improvement_pp:.1f}pp '
        f'(target: {TARGET_IMPROVEMENT_PP:.0f}pp).'
    )
    print(
        'Possible causes:\n'
        '  • Synthetic sessions may not cover new behavioral patterns not present in real data.\n'
        '  • The CTGAN model may need more training epochs or a higher n_sessions.\n'
        '  • The minority class is sparse — consider SMOTE or class weighting adjustments.\n'
        'Proceeding with v2 export regardless (document and move forward).'
    )

## Cell 5 — Feature importance delta (v1 vs v2)

In [ ]:
v1_imp = pd.Series(v1_model.feature_importances_, index=_FEATURE_COLS, name='v1')
v2_imp = pd.Series(v2_model.feature_importances_, index=_FEATURE_COLS, name='v2')

# Normalise to [0, 1] for fair comparison across different total gains
v1_imp_norm = v1_imp / v1_imp.sum()
v2_imp_norm = v2_imp / v2_imp.sum()

imp_df = pd.DataFrame({'v1': v1_imp_norm, 'v2': v2_imp_norm})
imp_df['delta'] = imp_df['v2'] - imp_df['v1']
imp_df = imp_df.sort_values('v2', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

imp_df[['v1', 'v2']].plot.barh(ax=axes[0], title='Normalised feature importance: v1 vs v2')
axes[0].invert_yaxis()
axes[0].set_xlabel('Relative importance')

imp_df['delta'].sort_values().plot.barh(
    ax=axes[1], title='Importance delta (v2 − v1)', color='#6366f1'
)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Delta')

plt.tight_layout()
plt.savefig('../docs/feature_importance_v1_vs_v2.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nFeature importance comparison (normalised):')
display(imp_df.round(4))

## Cell 6 — Save v2 model

In [ ]:
V2_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(v2_model, V2_MODEL_PATH)
print(f'v2 model saved → {V2_MODEL_PATH}')

# Verify the saved model produces identical predictions
v2_reloaded = joblib.load(V2_MODEL_PATH)
prob_check = v2_reloaded.predict_proba(X_test[:5])[:, 1]
print(f'Reload check (first 5 test probs): {np.round(prob_check, 4)}')

## Cell 7 — Generate model card

In [ ]:
training_date = datetime.now(tz=timezone.utc).strftime('%Y-%m-%d')
real_rows  = int((df_train['is_synthetic'] == 0).sum()) if 'is_synthetic' in df_train.columns else len(df_train)
synth_rows = int((df_train['is_synthetic'] == 1).sum()) if 'is_synthetic' in df_train.columns else 0

model_card = f"""# Model Card — LightGBM Lead Scorer v2

## Overview

Phase 18 augmented retrain of the Phase 11 LightGBM lead scoring model.
Training corpus combines real Retailrocket sessions with CTGAN-generated
synthetic sessions to improve recall on the minority conversion class.

## Training Data

| Source       | Rows       |
|--------------|------------|
| Real (Retailrocket) | {real_rows:,} |
| Synthetic (CTGAN)   | {synth_rows:,} |
| **Total train**     | **{len(df_train):,}** |
| Test (held-out 20%) | {len(df_test):,} |

## Features

{chr(10).join(f'- `{f}`' for f in _FEATURE_COLS)}

## Hyperparameters

- Algorithm: LightGBM binary classifier
- n_estimators: 400
- learning_rate: 0.05
- num_leaves: 63
- min_child_samples: 20
- scale_pos_weight: {scale_pos_weight:.1f}
- CV: 5-fold stratified
- random_state: {RANDOM_STATE}

## Performance vs v1 Baseline

| Metric          | v1 (baseline) | v2 (augmented) | Delta    |
|-----------------|---------------|----------------|----------|
| ROC-AUC         | {v1_auc:.4f}      | {v2_auc:.4f}       | {(v2_auc-v1_auc):+.4f}  |
| Recall@10%      | {v1_recall_k:.4f}      | {v2_recall_k:.4f}       | {(v2_recall_k-v1_recall_k):+.4f}  |
| Precision@10%   | {v1_precision_k:.4f}      | {v2_precision_k:.4f}       | {(v2_precision_k-v1_precision_k):+.4f}  |

## Known Limitations

- `max_scroll_pct` is `NULL` for all Retailrocket rows — NaN preserved for LightGBM.
- `search_count` is always 0 for Retailrocket rows.
- Synthetic sessions from CTGAN may not perfectly replicate edge-case behavioral patterns.
- Model is calibrated for e-commerce sessions; not suitable for other domains.

## Artifacts

- Model: `models/lead_scorer_lgbm_v2.pkl`
- Training data: `data/augmented_training.parquet` (gitignored)
- Feature importance chart: `docs/feature_importance_v1_vs_v2.png`

## Training Date

{training_date} UTC

## Usage

```python
from src.scoring.ml_scorer import MLScorer

# Load via registry (after make select-model VERSION=v2)
scorer = MLScorer()

# Or load v2 explicitly
scorer = MLScorer(model_version='v2')
scores = scorer.predict(session_df)
```
"""

MODEL_CARD_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_CARD_PATH.write_text(model_card, encoding='utf-8')
print(f'Model card written → {MODEL_CARD_PATH}')
print()
print(model_card)